# 01 - Generate Synthetic Agreement Data

This notebook generates synthetic contract PDFs **with amendments** and simulates scanned documents.

## Pipeline
1. Generate clean PDFs (5 base contracts + 4 amendments, EN and PL)
2. Apply scan simulation (rotation, noise, contrast reduction)
3. Verify outputs

## Amendment-Aware Design
Amendments override specific clauses from the base contract. The filename convention
encodes metadata for ingestion: `{nn}_{contract_id}_{source_type}_{lang}_v{version}_{effective_date}.pdf`

In [ ]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path

## Step 1: Generate Agreement PDFs with Amendments

We use `fpdf2` with DejaVu Sans font (Unicode support for Polish characters).

**Base contracts (5):**
| Contract ID | Type | Language | Effective Date |
|---|---|---|---|
| ITSVC-001 | IT Service Agreement | EN | 2025-01-15 |
| NDA-001 | Mutual NDA | EN | 2025-03-01 |
| LEASE-001 | Office Lease | PL | 2025-02-10 |
| SLA-001 | Cloud SLA | EN | 2025-04-01 |
| EMP-001 | Employment Contract | PL | 2025-05-01 |

**Amendments (4):**
| Contract ID | Amendment | Effective Date | Key Changes |
|---|---|---|---|
| ITSVC-001 | v2 | 2025-07-01 | Rate increase (+15%), team 3→5 |
| ITSVC-001 | v3 | 2026-01-01 | Rate increase (+10%), new AI/ML tier, bi-weekly invoicing |
| LEASE-001 | v2 | 2025-09-01 | Rent 25k→28.5k PLN, 5 parking spaces |
| SLA-001 | v2 | 2025-10-01 | P1 response 15→5 min, credit cap 30→50% |

In [ ]:
%run ../scripts/generate_agreements.py

In [ ]:
# Verify generated files
agreements = sorted(Path('../data/agreements').glob('*.pdf'))
for p in agreements:
    print(f'{p.name:45s} {p.stat().st_size / 1024:.1f} KB')

## Step 2: Simulate Scanned Documents

Each PDF is converted to images and degraded to look like a scan:
- Slight rotation (±1.5°)
- Gaussian noise
- Reduced contrast and brightness
- Subtle blur

In [ ]:
%run ../scripts/simulate_scans.py

In [ ]:
# Verify scanned files (should be much larger - image-based PDFs)
scans = sorted(Path('../data/scans').glob('*.pdf'))
for p in scans:
    print(f'{p.name:45s} {p.stat().st_size / 1024:.1f} KB')

## Summary

We now have:
- `data/agreements/` — 9 clean PDFs (5 bases + 4 amendments)
- `data/scans/` — 9 scan-simulated PDFs ready for OCR/ingestion

Each filename encodes: `contract_id`, `source_type`, `language`, `version`, `effective_date`.

**Next:** Run `02_ingestion.ipynb` to embed and index these documents in Pinecone.